## Phase 1 - Defining the Agents

### 1. Import AgentPy

In [2]:
# Run this cell first to import the library

%pip install agentpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 2.1 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [agentpy]m2/4 [SALib]rocess]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import agentpy as ap

### 2.The MachineAgent

In [5]:
class MachineAgent(ap.Agent):

    # The setup method acts as our constructor in AgentPy
    def setup(self, capability="Drilling"):
        # Save the machine's capability
        self.capability = capability

        # Initialize an empty list to act as our message inbox
        self.inbox = []

        # Print a startup message confirming the agent's ID and capability
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

### 3. The OrderAgent

In [6]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        # Save what operation this order needs
        self.required_operation = required_operation

        # Initialize its own empty inbox
        self.inbox = []

        # Print a startup message
        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

#### 4. Testing the Agents

In [7]:
# 1. Create a blank base model to house our agents
dummy_model = ap.Model()

# 2. Spawn one MachineAgent and pass the capability argument
machine1 = MachineAgent(dummy_model, capability="Drilling")

# 3. Spawn one OrderAgent and pass the required_operation argument
order1 = OrderAgent(dummy_model, required_operation="Drilling")

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling


## Phase 2 - Building the Model Environment

### This is where AgentPy really simplifies things compared to JADE. Instead of relying on a complex Directory Facilitator (the Yellow Pages) to discover other agents, AgentPy uses a centralized Model to orchestrate everything. The Model acts as the environment that holds all the agents together. This means that any agent can easily access any other agent through the Model, without needing to go through a separate discovery process. The Model provides a straightforward way for agents to find and interact with each other, making the implementation much simpler and more intuitive.

### 5. The Model Environment

In [8]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # 1. Instantiate agents using ap.AgentList
        # This creates lists containing 1 agent of each type, passing the required arguments
        self.machines = ap.AgentList(self, 1, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # 2. Service Discovery: Give the OrderAgent its "Contact List"
        # We grab the first (and only) order agent at index 0 and hand it the list of machines
        self.orders[0].target_machines = self.machines

        print("\n--- Model Setup Complete ---")

# 3. Execution Block
if __name__ == '__main__':
    # Instantiate the model
    model = ManufacturingModel()

    # Run the setup to spawn the agents and trigger their print statements
    model.setup()

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling

--- Model Setup Complete ---


## Phase 3 - Enabling Agent Communication

### By using simple state variables (like self.state), we can easily control whether an agent does something every single tick (like listening) or just once (like sending an initial request).

#### 6. Updated MachineAgent with a step method to listen for requests

In [9]:
class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # Because there is no state check here, this will print on EVERY tick
        print(f"Machine Agent {self.id} is waiting for work...")

### 7. Updated OrderAgent with a state variable to control when it sends its request

In [10]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []

        # --- PHASE 3: State Tracking ---
        self.state = 'seeking_machine'

        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

    # --- PHASE 3: The Step Method ---
    def step(self):
        # This acts like a JADE OneShotBehaviour because we change the state immediately after!
        if self.state == 'seeking_machine':
            print(f"Order Agent {self.id} is looking for machines...")

            # Change the state so this block never runs again
            self.state = 'waiting_for_bids'

### 8. Updated Model and Execution

#### To actually see the time steps in action, we need to update our model to tell the agents to take their steps, and then use AgentPy's built-in model.run() function instead of just calling setup().

In [11]:
class ManufacturingModel(ap.Model):

    def setup(self):
        self.machines = ap.AgentList(self, 1, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        # Service Discovery configuration
        self.orders[0].target_machines = self.machines
        print("\n--- Model Setup Complete ---\n")

    # --- PHASE 3: Model Step ---
    def step(self):
        # On every tick of the simulation, tell our agent lists to run their step() methods
        self.machines.step()
        self.orders.step()

# Execution Block
if __name__ == '__main__':
    model = ManufacturingModel()

    # Run the model for exactly 2 time steps to see our state logic work!
    # (We hide the default AgentPy output by storing it in 'results')
    results = model.run(steps=2, display=False)

Machine Agent 1 is ready. Registered capability: Drilling
Order Agent 2 is ready. Required operation: Drilling

--- Model Setup Complete ---

Machine Agent 1 is waiting for work...
Order Agent 2 is looking for machines...
Machine Agent 1 is waiting for work...


## Phase 4 - Implementing the Contract Net Protocol

### 9.Updated MachineAgent (The Receiver)

#### We are replacing the basic print statement with a loop that checks the self.inbox. It reads the dictionary, prints the result, and deletes the message so it isn't read twice.

In [12]:
class MachineAgent(ap.Agent):

    def setup(self, capability="Drilling"):
        self.capability = capability
        self.inbox = []
        print(f"Machine Agent {self.id} is ready. Registered capability: {self.capability}")

    # --- PHASE 4: Reading the Inbox ---
    def step(self):
        # We iterate over a copy of the list [:] so we can safely .remove() items from the real list
        for msg in self.inbox.copy():

            # Check the intent (performative) of the message
            if msg["performative"] == "CFP":
                sender_id = msg['sender'].id
                print(f">>> Machine {self.id} received a job request from Order {sender_id}.")
                print(f"    Message content: {msg['content']}")

            # Clear the message from the inbox after reading it
            self.inbox.remove(msg)

### 10. Updated OrderAgent (The Sender)

#### We are updating the seeking_machine state to construct our Python dictionary message and append it directly to the target machines.

In [13]:
class OrderAgent(ap.Agent):

    def setup(self, required_operation="Drilling"):
        self.required_operation = required_operation
        self.inbox = []
        self.state = 'seeking_machine'
        print(f"Order Agent {self.id} is ready. Required operation: {self.required_operation}")

    # --- PHASE 4: Sending the Message ---
    def step(self):
        if self.state == 'seeking_machine':
            print(f"<<< Order {self.id} is broadcasting Call for Proposals (CFP)...")

            # 1. Construct the simulated FIPA-ACL message
            message = {
                "performative": "CFP",
                "sender": self,  # We pass a reference to ourselves so they can reply!
                "content": "Job_Request"
            }

            # 2. Drop the message into the inbox of every capable machine
            for machine in self.target_machines:
                machine.inbox.append(message)

            # 3. Change state to wait for replies
            self.state = 'waiting_for_bids'

### 11. Updated Model and Execution

#### To really see the broadcasting work, let's bump the number of machines up to two in our model setup, and ensure we call model.run(steps=2).

In [14]:
class ManufacturingModel(ap.Model):

    def setup(self):
        # Let's spawn 2 machines to prove the OrderAgent can broadcast!
        self.machines = ap.AgentList(self, 2, MachineAgent, capability="Drilling")
        self.orders = ap.AgentList(self, 1, OrderAgent, required_operation="Drilling")

        self.orders[0].target_machines = self.machines
        print("\n--- Model Setup Complete ---\n")

    def step(self):
        # Order dictates execution sequence per tick.
        # We run orders first so they send the message, then machines so they read it in the same tick.
        self.orders.step()
        self.machines.step()

# Execution Block
if __name__ == '__main__':
    model = ManufacturingModel()

    # Run the model for 2 steps
    results = model.run(steps=2, display=False)

Machine Agent 1 is ready. Registered capability: Drilling
Machine Agent 2 is ready. Registered capability: Drilling
Order Agent 3 is ready. Required operation: Drilling

--- Model Setup Complete ---

<<< Order 3 is broadcasting Call for Proposals (CFP)...
>>> Machine 1 received a job request from Order 3.
    Message content: Job_Request
>>> Machine 2 received a job request from Order 3.
    Message content: Job_Request
